In [0]:
select current_catalog(), current_schema();

current_catalog(),current_schema()
joins,tasks2


In [0]:
create schema if not exists joins.tasks2;
use joins.tasks2;

In [0]:
-- domains table
create table if not exists domains(
  id int primary key,
  domain_name string,
  color_code string
)
using delta;

In [0]:
-- trainees table 
create table if not exists trainees(
  id int primary key,
  full_name string,
  email string,
  domain_id int,
  foreign key(domain_id) references domains(id)
)
using delta;

In [0]:
-- assessments table
create table if not exists assessments(
  id int primary key,
  trainee_id int,
  score int,
  interview_link string,
  report_link string,
  potential string,
  selection_status string,
  foreign key(trainee_id) references trainees(id)
)
using delta;

In [0]:
insert into domains values
(1, 'MERN', 'bg-info'),
(2, 'PHP', 'bg-primary'),
(3, 'QA', 'bg-warning'),
(4, 'DevOps', 'bg-dark'),
(5, 'Data Engineering', 'bg-danger');

insert into trainees values
(101, 'Rehan Raza', 'rehan@kadel.com', 1),
(102, 'Hitesh Suthar', 'hitesh@kadel.com', 2),
(103, 'Gargi Sharma', 'gargi@kadel.com', 3),
(104, 'Aditya Chhipa', 'aditya@kadel.com', 1),
(105, 'Neeraj Vyas', 'neeraj@kadel.com', 4);

insert into assessments values
(1, 101, 92, 'https://meet.google.com/xyz', 'https://drive.com/rehan', 'Excellent', 'Selected'),
(2, 102, 78, 'https://meet.google.com/abc', 'https://drive.com/hitesh', 'Good', 'Waitlist'),
(3, 103, 85, 'https://meet.google.com/def', 'https://drive.com/gargi', 'Excellent', 'Selected'),
(4, 104, 65, 'https://meet.google.com/ghi', 'https://drive.com/aditya', 'Average', 'Pending'),
(5, 105, 88, 'https://meet.google.com/jkl', 'https://drive.com/neeraj', 'Excellent', 'Selected');

num_affected_rows,num_inserted_rows
5,5


**Q1. Write a query to find all trainees whose assessment_score is higher than the
average score of all trainees.**

In [0]:
select t.full_name, a.score
from trainees t
join assessments a 
on t.id = a.trainee_id
where a.score > (
  select avg(score) from assessments
);

full_name,score
Rehan Raza,92
Gargi Sharma,85
Neeraj Vyas,88


**Q2. List the names of trainees in the 'MERN' domain who scored higher than the highest
scorer in the 'PHP' domain.**

In [0]:
select t.full_name, a.score 
from trainees t
     join domains d 
     on t.domain_id = d.id
     join assessments a 
     on t.id = a.trainee_id
where a.score > (
  select max(a1.score) 
  from trainees t1
       join domains d1
       on t1.domain_id = d1.id
       join assessments a1 
       on t1.id = a1.trainee_id
  where d1.domain_name = 'PHP'
)
and d.domain_name = 'MERN'

full_name,score
Rehan Raza,92


**Q3. Correlated: Find trainees who scored higher than the average score of their own
specific domain.**

In [0]:
select t.full_name, a.score
from trainees t
join assessments a
on t.id = a.trainee_id
where a.score > (
  select avg(a1.score)
  from assessments a1
  join trainees t1
  on a1.trainee_id = t1.id
  where t1.domain_id = t.domain_id
)

full_name,score
Rehan Raza,92


**Q4. Use a subquery with NOT IN to find domains that currently have no trainees assigned
to them.**

In [0]:
select d.domain_name 
from domains d
where d.id not in (
  select t.domain_id from trainees t
)

domain_name
Data Engineering


**Q5. Retrieve the trainee_name and domain for the trainee who has the maximum score
in the entire table.**

In [0]:
select t.full_name, a.score ,d.domain_name
from trainees t 
join assessments a  
on t.id = a.trainee_id
join domains d
on d.id = t.domain_id
where a.score = (
  select max(score) from assessments
)

full_name,score,domain_name
Rehan Raza,92,MERN


**Q6. Use EXISTS to find all domains that have at least one trainee marked as 'Excellent'.**

In [0]:
select d.domain_name
from domains d
where exists (
  select * from trainees t
  join assessments a
  on t.id = a.trainee_id
  where d.id = t.domain_id
  and a.potential = 'Excellent'
)

domain_name
MERN
QA
DevOps


**Q7. Find the second-highest assessment_score using a nested subquery.**

In [0]:
select max(a1.score) as second_highest_assessment_score
from assessments a1
where a1.score < (
  select max(a2.score) from assessments a2
)

second_highest_assessment_score
88


**Q8. Select trainees whose scores are within 5 points of the overall average (using a
subquery in the WHERE clause).**

In [0]:
select t.full_name, a.score, d.domain_name
from trainees t
join domains d 
on d.id = t.domain_id
join assessments a 
on t.id = a.trainee_id
where a.score between (select avg(score) from assessments)-5 and (select avg(score) from assessments)+5
order by a.score desc

full_name,score,domain_name
Gargi Sharma,85,QA
Hitesh Suthar,78,PHP


In [0]:
select t.full_name, a.score, d.domain_name
from trainees t
join domains d 
on d.id = t.domain_id
join assessments a 
on t.id = a.trainee_id
where a.score > (select avg(score) from assessments)-5 and a.score < (select avg(score) from assessments)+5
order by a.score desc

full_name,score,domain_name
Gargi Sharma,85,QA
Hitesh Suthar,78,PHP


**Q9. Create a CTE named HighScorers that selects trainees with scores > 80, then select
only the 'QA' trainees from that CTE.**

In [0]:
with highScorers as (
  select t.full_name, a.score, d.domain_name
  from trainees t
  join domains d 
  on d.id = t.domain_id
  join assessments a 
  on t.id = a.trainee_id
  where a.score > 80
)
select * from highscorers where domain_name = 'QA'

full_name,score,domain_name
Gargi Sharma,85,QA


**Q10. Use a CTE to calculate the average score per domain, then join it back to the main
table to show how much each trainee is above/below their domain average.**

In [0]:
with cte as (
  select t.domain_id, avg(a.score) as total_avgscore
  from trainees t
  join assessments a
  on t.id = a.trainee_id
  group by t.domain_id
)
select t.full_name, d.domain_name, a.score, da.total_avgscore,
da.total_avgscore - a.score as difference
from cte da
join trainees t
  on t.domain_id = da.domain_id
join assessments a
  on t.id = a.trainee_id
join domains d
  on t.domain_id = d.id

full_name,domain_name,score,total_avgscore,difference
Rehan Raza,MERN,92,78.5,-13.5
Hitesh Suthar,PHP,78,78.0,0.0
Gargi Sharma,QA,85,85.0,0.0
Aditya Chhipa,MERN,65,78.5,13.5
Neeraj Vyas,DevOps,88,88.0,0.0


**Q11. Readability: Rewrite a complex 3-table join using two separate CTEs to make the
final SELECT statement cleaner.**

In [0]:
with cte1 as(
  select t.id, t.full_name, d.domain_name
  from trainees t
  join domains d
  on t.domain_id = d.id
),
cte2 as(
  select
  a.id, a.trainee_id,a.score
  from assessments a
)
select c1.full_name, c1.domain_name, c2.score
from cte1 c1
join cte2 c2
on c1.id = c2.trainee_id

full_name,domain_name,score
Rehan Raza,MERN,92
Hitesh Suthar,PHP,78
Gargi Sharma,QA,85
Aditya Chhipa,MERN,65
Neeraj Vyas,DevOps,88


In [0]:
with cte1 as(
  select t.id, t.full_name, d.domain_name
  from trainees t
  join domains d
  on t.domain_id = d.id
),
cte2 as(
  select *,
  a.id, a.trainee_id,a.score
  from cte1 c
  join assessments a
  on c.id = a.trainee_id
)
select * from cte2

id,full_name,domain_name,id,trainee_id,score,interview_link,report_link,potential,selection_status,id,trainee_id,score
101,Rehan Raza,MERN,1,101,92,https://meet.google.com/xyz,https://drive.com/rehan,Excellent,Selected,1,101,92
102,Hitesh Suthar,PHP,2,102,78,https://meet.google.com/abc,https://drive.com/hitesh,Good,Waitlist,2,102,78
103,Gargi Sharma,QA,3,103,85,https://meet.google.com/def,https://drive.com/gargi,Excellent,Selected,3,103,85
104,Aditya Chhipa,MERN,4,104,65,https://meet.google.com/ghi,https://drive.com/aditya,Average,Pending,4,104,65
105,Neeraj Vyas,DevOps,5,105,88,https://meet.google.com/jkl,https://drive.com/neeraj,Excellent,Selected,5,105,88


**Q12. Create a CTE that ranks trainees by score and then select only the top 3 trainees.**

In [0]:
with cte as (select t.full_name,a.score, dense_rank() over(order by a.score desc) as rank
from trainees t
join assessments a
on t.id = a.trainee_id)
select * from cte where rank <= 3;

full_name,score,rank
Rehan Raza,92,1
Neeraj Vyas,88,2
Gargi Sharma,85,3


**Q13. Write a CTE that identifies "Top Performers" (Score > 90) and "Average Performers"
(Score 50-90), then count how many are in each category.**

In [0]:
with cte as (
  select a.score,
  case
  when a.score > 90 then "Top Performers"
  when a.score between 50 and 90 then "Average Performers"
  end as performance
  from assessments a
)
select performance, count(score) from cte
group by performance;

performance,count(score)
Top Performers,1
Average Performers,4


**Q14. Use a CTE to find the domain with the highest total number of 'Excellent' trainees.**

In [0]:
with cte as(
  select
  d.domain_name,a.potential
  from domains d
  join trainees t
  on d.id = t.domain_id
  join assessments a
  on t.id = a.trainee_id
  WHERE a.potential = 'Excellent'
)
select domain_name ,count(potential) from cte
group by domain_name
order by count(potential) desc;

domain_name,count(potential)
DevOps,1
MERN,1
QA,1


**Q15. Recursive (Advanced): If you had a table of Managers and Subordinates, how
would you use a Recursive CTE to show a full organizational hierarchy?**

In [0]:
-- exception case
with cte1 as(
  select t.id, t.full_name, d.domain_name
  from trainees t
  join domains d
  on t.domain_id = d.id
),
cte2 as(
  select
  a.id, a.trainee_id,a.score
  from assessments a
)
select c1.full_name, c1.domain_name, c2.score
from cte1 c1
join cte2 c2
on c1.id = c2.trainee_id

**Q16. Use a CTE to temporarily store a list of "Flagged" interview links (links that are
NULL or empty) and count them.**

In [0]:
with Flagged as (
  select interview_link from assessments
  where interview_link is null
  or interview_link = ''
)
select count(*) from Flagged

count(*)
0


**Q17. Combine a list of trainees from the mern_batch table and the php_batch table
using UNION (removing duplicates).**

In [0]:
select t.full_name, d.domain_name
from trainees t
join domains d  
on t.domain_id = d.id
where domain_name = 'MERN'
union
select t.full_name, d.domain_name
from trainees t
join domains d  
on t.domain_id = d.id
where domain_name = 'PHP'

full_name,domain_name
Aditya Chhipa,MERN
Rehan Raza,MERN
Hitesh Suthar,PHP


**Q18. What is the difference between UNION and UNION ALL? Write a query illustrating
when you would use UNION ALL for your assessment logs.**

In [0]:
-- UNION vs UNION ALL (Easy Language)
-- Point	     |         UNION	           |      UNION ALL
-- Duplicate	 |   ❌ remove karta hai	    | ✔ allow karta hai
-- Performance |	        slow	           |       fast
-- Use case	   |       clean unique data	 |    raw data / logs


-- Example 
-- Data:
-- MERN → Rehan, Aditya  
-- PHP → Hitesh, Rehan

-- 👉 UNION:

-- Rehan
-- Aditya
-- Hitesh

-- 👉 UNION ALL:

-- Rehan
-- Aditya
-- Hitesh
-- Rehan   ← duplicate bhi aaya

In [0]:
select t.full_name, d.domain_name
from trainees t
join domains d  
on t.domain_id = d.id
where domain_name = 'MERN'
UNION ALL
select t.full_name, d.domain_name
from trainees t
join domains d  
on t.domain_id = d.id
where domain_name = 'MERN'

full_name,domain_name
Rehan Raza,MERN
Aditya Chhipa,MERN
Rehan Raza,MERN
Aditya Chhipa,MERN


**Q19. Use INTERSECT to find trainees who are present in both the 'Internal Assessment' list
and the 'External Certification' list.**

In [0]:
SELECT * FROM TABLE
INTERSECT
SELECT * FROM TABLE

**Q20. Use EXCEPT (or MINUS) to find trainees who passed the 'Initial Screening' but have
not yet taken the 'Final Technical Interview'.**

In [0]:
SELECT * FROM TABLE
EXCEPT
SELECT * FROM TABLE

**Q21. Write a query that returns a single column of all unique names from three different
project tables using UNION.**

In [0]:
select domain_name from domains
union
select full_name from trainees
union
select potential from assessments;

domain_name
MERN
PHP
QA
DevOps
Data Engineering
Rehan Raza
Hitesh Suthar
Gargi Sharma
Aditya Chhipa
Neeraj Vyas


**Q22. Combine two queries where the first selects "Excellent" trainees and the second
selects "Good" trainees, adding a hardcoded column 'Priority' to each.**

In [0]:
select t.full_name, a.potential, "High" as priority
from trainees t
join assessments a
on t.id = a.trainee_id
where a.potential = "Excellent"
union
select t.full_name, a.potential, "Medium" as priority
from trainees t
join assessments a
on t.id = a.trainee_id
where a.potential = "Good"

full_name,potential,priority
Gargi Sharma,Excellent,High
Neeraj Vyas,Excellent,High
Rehan Raza,Excellent,High
Hitesh Suthar,Good,Medium


**Q23. Find the domains that are common between your current_trainings table and
your upcoming_projects table using INTERSECT.**

In [0]:
select domain_name from current_trainings
intersect
select domain_name from upcoming_projects

**Q24. List all trainees who have submitted an Assessment_Link but have not yet
provided an Interview_Link using EXCEPT.**

In [0]:
-- Assessment_Link = report_link
select trainee_id
from assessments 
where report_link is not null
except
select trainee_id
from assessments
where interview_Link is not null

trainee_id


**Q25. The Mix: Write a query that uses a CTE to gather data, and then performs a UNION
between two different filtered versions of that CTE.**

In [0]:
with cte as(
  select t.full_name, d.domain_name
  from trainees t
  join domains d
  on t.domain_id = d.id
)
select * from cte where domain_name = 'MERN'
union
select * from cte where domain_name = 'PHP'

full_name,domain_name
Aditya Chhipa,MERN
Rehan Raza,MERN
Hitesh Suthar,PHP
